# Filter showcase -- P3 LatestVectorClassification

Presentation PNGs of what four classification filters do on one page. Each filter gets its own sequence: the
original, what the filter groups or flags, what it drops and what it keeps, and the output after that step.
Nothing is re-implemented. The page goes through `classify_vectors(keep_steps=True)`, and each image reads that
run's per-step categories or calls the functions those steps call. Logic: `filter_showcase_helpers.py`.

- **1 Collinear**: original -> collinear groups -> one random group per global direction (full page,
  plus `03_global_angles.txt`) -> drawing / not-drawing groups -> after.
- **2 Pattern lattice**: original -> 5 x (3x3 similar vectors) -> 5 largest lattice groups -> large lattices
  dropped / kept / remaining -> bucket-rule drops -> after.
- **3 Length outliers**: 3 random clusters with outliers. Original -> parallel groups -> outliers -> after.
- **4 Crossings**: 3 random clusters with both dominant-grid drops and off-grid flagged lines. Original ->
  flagged lines + crossing dots -> dominant vs off-grid -> dropped -> after.

Page-wide images (1, 2) are auto-zoomed per filter. The zoom window is the square of side `ZOOM_FRAC` x the
page's short side that holds the most of both dropped and kept vectors. `CLIP` overrides it.
The original is the real page. "After" images use the vectors' real paint, as Phase 4 draws them.
Highlights are drawn over a light-grey context.

Output: `outputs/filter_showcase/<ts>__<stem>_p<N>/`.

## 0 - Config

In [ ]:
PDF_PATH = None  # None -> first PDF under references/
PAGE = 0
SEED = 0  # random picks (representatives, 3x3 samples, clusters) are reproducible
SIDE_PX = 1600  # long side of every image, px
ZOOM_FRAC = 0.25  # auto-zoom window side, as a fraction of the page's short side
CLIP = None  # (x0, y0, x1, y1) unrotated page space -> overrides the auto-zoom for 1 and 2
N_CLUSTERS = 3  # clusters for length outliers / crossings
TOP_K = 5  # similarity buckets / lattice groups for the pattern filter

## 1 - Bootstrap

In [ ]:
import os, sys
from pathlib import Path

_root = os.path.abspath(os.path.join(".."))
if _root not in sys.path:
    sys.path.insert(0, _root)

from IPython.display import Image, display

from notebooks._helpers import filter_showcase_helpers as h

if PDF_PATH is None:
    refs = sorted((Path(_root) / "references").glob("*.pdf"))
    assert refs, "no PDFs found under references/"
    PDF_PATH = str(refs[0])
print(PDF_PATH, "page", PAGE)

r = h.run(PDF_PATH, PAGE)
sc = h.Showcase(r, side_px=SIDE_PX, seed=SEED)
print(
    f"{len(r.vectors)} vectors, {len(r.buckets)} buckets, global angles: "
    + ", ".join(f"{a:.1f}" for a in r.result.global_angles)
)
print("output:", sc.root)


def show(prefix: str, width: int = 900) -> None:
    for section, paths in sc.saved.items():
        if section.startswith(prefix):
            for path in paths:
                print(f"{section}/{path.name}")
                display(Image(filename=str(path), width=width))

## 2 - Collinear drawing

In [ ]:
h.showcase_collinear(sc, frac=ZOOM_FRAC, clip=CLIP)
print((sc.root / "1_collinear" / "03_global_angles.txt").read_text())
show("1_collinear")

## 3 - Pattern lattice

In [ ]:
h.showcase_pattern(sc, top_k=TOP_K, frac=ZOOM_FRAC, clip=CLIP)
show("2_pattern")

## 4 - Length outliers

In [ ]:
h.showcase_length_outliers(sc, n=N_CLUSTERS)
show("3_length_outliers")

## 5 - Crossings

In [ ]:
h.showcase_crossings(sc, n=N_CLUSTERS)
show("4_crossings")

## 6 - Output folder

In [ ]:
print(sc.root)
for section, paths in sc.saved.items():
    print(f"{section}: {len(paths)} images")